In [1]:
import pandas as pd
import torch
import librosa
import os
from transformers import WhisperFeatureExtractor, WhisperModel
from tqdm.notebook import tqdm # visualization
import numpy as np 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader # for training
from sklearn.utils.class_weight import compute_class_weight # class weights
from sklearn.model_selection import train_test_split # split data
from sklearn.preprocessing import LabelEncoder #transform labels
import matplotlib.pyplot as plt # visualization
import seaborn as sns # visualization
from sklearn.metrics import confusion_matrix, classification_report # visualization
import copy # copy and save model 
import math
import gc

In [2]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
device = torch.device(device)
print("Using device:", device)

Using device: cuda


In [4]:
label_cols = ['Prolongation', 'Block', 'SoundRep', 'WordRep', 'Interjection']
print("Loading data...")
X_raw = np.load('whisper_embeddings2k.npy') 

# Slice just to make sure 150 embedded dimension.
X = X_raw[:, :150, :] 

# Free up the old memory immediately
print(f"Original Shape: {X_raw.shape} | Optimized Shape: {X.shape}")
del X_raw
gc.collect() 

df = pd.read_csv('SEP-28k_with_paths2k.csv')
y_text = df[label_cols]

label_encoder = LabelEncoder()
y = df[label_cols].to_numpy(dtype=np.float32)
num_classes = 5

indices = np.arange(len(df)) #to keep the indicies for later.

idx_temp, idx_test = train_test_split(
    indices, test_size=0.2, random_state=42
)

idx_train, idx_val = train_test_split(
    idx_temp, test_size=0.25, random_state=42
)

X_train, X_val, X_test = X[idx_train], X[idx_val], X[idx_test]
y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]



# # Convert to Tensors (Directly onto CPU first, then DataLoaders handle device move)
# X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
# y_train_tensor = torch.tensor(y_train, dtype=torch.long)
# X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
# y_val_tensor = torch.tensor(y_val, dtype=torch.long)

# train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
# val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32, shuffle=False)

# print(f"🎉 Memory Cleaned. Data Splits -> Train: {len(X_train)} | Val: {len(X_val)}")

Loading data...
Original Shape: (12018, 150, 512) | Optimized Shape: (12018, 150, 512)


In [5]:
batch_size = 32

train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [6]:
# Cnn + transformer

class TemporalCNNEncoder(nn.Module):
    def __init__(self):
        super(TemporalCNNEncoder, self).__init__()
        # Whisper base dim is 512. We treat these as "Channels" for the CNN
        self.conv1 = nn.Conv1d(in_channels=512, out_channels=256, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(256)
        self.conv2 = nn.Conv1d(in_channels=256, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu = nn.GELU()
        self.pool = nn.MaxPool1d(kernel_size=2) 

    def forward(self, x):
        # x enters as [Batch, 150, 512]. Conv1d: [Batch, Channels, Time]
        x = x.transpose(1, 2) 
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        return x.transpose(1, 2) # Returns [Batch, 75, 128] for Transformer
    

class TransformerBrain(nn.Module):
    def __init__(self, num_labels, d_model=128, seq_length=75):
        super(TransformerBrain, self).__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_length, d_model) * 0.01)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=512, dropout=0.3, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.fc = nn.Linear(d_model, num_labels)

    def forward(self, x):
        x = x + self.pos_embedding
        x = self.transformer(x)
        x = x.mean(dim=1) 
        return self.fc(x)
    

class EndToEndStutterModel(nn.Module):
    def __init__(self, num_labels):
        super(EndToEndStutterModel, self).__init__()
        self.cnn = TemporalCNNEncoder()
        self.transformer = TransformerBrain(num_labels)

    def forward(self, x):
        return self.transformer(self.cnn(x))

In [7]:
num_labels = len(label_cols)
model = EndToEndStutterModel(num_labels=num_labels).to(device)

In [8]:
# training and weights 
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

device = torch.device(device)
print("Using device:", device)

criterion = nn.BCEWithLogitsLoss() #for multilabel classification
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)

patience = 15
best_val_loss = float('inf')
patience_counter = 0
best_model_weights = copy.deepcopy(model.state_dict())

Using device: cuda


In [9]:

from sklearn.metrics import f1_score

epochs = 50
threshold = 0.5

for epoch in range(1, epochs + 1):

    # TRAIN

    model.train()
    train_loss = 0.0

    for Xb, yb in train_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(Xb)                # [B, 5]
        loss = criterion(logits, yb)      # BCEWithLogitsLoss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # VALIDATION
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb = Xb.to(device)
            yb = yb.to(device)

            logits = model(Xb)
            loss = criterion(logits, yb)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).int()

            all_preds.append(preds.cpu())
            all_labels.append(yb.cpu())

    val_loss /= len(val_loader)

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    val_micro = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    val_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)


    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Micro-F1: {val_micro:.4f} | "
        f"Val Macro-F1: {val_macro:.4f}"
    )
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_wts = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1
    if epochs_no_improve >= patience:
        print(f"Early stopping triggered after {epoch} epochs")
        break



Epoch 01 | Train Loss: 0.3112 | Val Loss: 0.2953 | Val Micro-F1: 0.6543 | Val Macro-F1: 0.6392
Epoch 02 | Train Loss: 0.2606 | Val Loss: 0.2900 | Val Micro-F1: 0.6828 | Val Macro-F1: 0.6706
Epoch 03 | Train Loss: 0.2437 | Val Loss: 0.2801 | Val Micro-F1: 0.6660 | Val Macro-F1: 0.6643
Epoch 04 | Train Loss: 0.2208 | Val Loss: 0.2976 | Val Micro-F1: 0.6658 | Val Macro-F1: 0.6304
Epoch 05 | Train Loss: 0.2059 | Val Loss: 0.3092 | Val Micro-F1: 0.6633 | Val Macro-F1: 0.6603
Epoch 06 | Train Loss: 0.1838 | Val Loss: 0.3197 | Val Micro-F1: 0.6636 | Val Macro-F1: 0.6534
Epoch 07 | Train Loss: 0.1647 | Val Loss: 0.3258 | Val Micro-F1: 0.6658 | Val Macro-F1: 0.6712
Epoch 08 | Train Loss: 0.1433 | Val Loss: 0.3788 | Val Micro-F1: 0.6445 | Val Macro-F1: 0.6420
Epoch 09 | Train Loss: 0.1256 | Val Loss: 0.3945 | Val Micro-F1: 0.6559 | Val Macro-F1: 0.6489
Epoch 10 | Train Loss: 0.1137 | Val Loss: 0.4081 | Val Micro-F1: 0.6620 | Val Macro-F1: 0.6475
Epoch 11 | Train Loss: 0.0939 | Val Loss: 0.4779 |

In [10]:
#saving model
MODEL_PATH = "full_model2k.pt"

torch.save(model.state_dict(), MODEL_PATH)

print(f"Model saved to {MODEL_PATH}")

Model saved to full_model2k.pt


In [11]:
from sklearn.metrics import f1_score

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoader
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)

# Evaluate
model.eval()
all_probs = []

with torch.no_grad():
    for batch_X, _ in test_loader:
        batch_X = batch_X.to(device)
        probs = torch.sigmoid(model(batch_X))
        all_probs.append(probs.cpu())

probs_np = torch.cat(all_probs, dim=0).numpy()



In [12]:
def clipName(row):
    return f"{row['Show']}_{row['EpId']}_{row['ClipId']}"

# Only test rows
df_test = df.iloc[idx_test].reset_index(drop=True)

clips_test = df_test.apply(clipName, axis=1)

probs_df = pd.DataFrame(probs_np, columns=label_cols)
probs_df.insert(0, "Clip", clips_test.values)

probs_df.to_csv("SEP28k-testprob_fullmodel.csv", index=False)

In [13]:
! python evaluate.py --reference SEP-28k-Extended_clips.csv --prediction SEP28k-testprob_fullmodel.csv --reference_threshold 2 --prediction_threshold 0.5                                  


# Block
## tp = 188 , tn = 1677 , fp = 245 , fn = 294
## precision = 0.434 , recall = 0.39 , f1 = 0.411 , accuracy = 0.776 , specificity = 0.873

# Interjection
## tp = 366 , tn = 1860 , fp = 93 , fn = 85
## precision = 0.797 , recall = 0.812 , f1 = 0.804 , accuracy = 0.926 , specificity = 0.952

# Prolongation
## tp = 321 , tn = 1771 , fp = 150 , fn = 162
## precision = 0.682 , recall = 0.665 , f1 = 0.673 , accuracy = 0.87 , specificity = 0.922

# SoundRep
## tp = 244 , tn = 1877 , fp = 148 , fn = 135
## precision = 0.622 , recall = 0.644 , f1 = 0.633 , accuracy = 0.882 , specificity = 0.927

# WordRep
## tp = 408 , tn = 1803 , fp = 128 , fn = 65
## precision = 0.761 , recall = 0.863 , f1 = 0.809 , accuracy = 0.92 , specificity = 0.934
